# 30 Survival: competing risks, parametric models, restricted means, adjusted curves, imputation

Real-data checks of the group "Survival". This notebook holds only simplydata: data read with `sdata.table`, the
words, the numbers read back with simplydata's own words (through `Result`), and the comparison with
`references/30_survival_more.csv`; each comparison prints its "Source:" with a link. No printed number matches these
checks, so `scripts/references/30_survival_more.py` runs the help-page or vignette example of the R function simplydata
calls once (survival, cmprsk, survRM2, adjustedCurves, mice). Data: R survival's mgus2 with the competing-risks set-up
of its vignette "compete" (time to plasma cell malignancy (PCM) or death), lung, ovarian. Tolerances: relative 1e-10
for closed formulas, 1e-6 for fits by iteration, 4 Monte Carlo standard errors for multiple imputation.

The numbers the survival charts draw (`draw_survival_curves`, `draw_risk_curves`) are read back with
`chart.get_numbers_drawn()` (section 7).

In [1]:
import simplydata as sdata

from reference_checks import Comparisons, Result, quiet

cmp = Comparisons("30_survival_more")
DATA = "../data/"
FETCHED = "../data/fetched/"  # the datasets this repository does not host (written by scripts/get_data.py)
mgus = sdata.table(cmp.data_file("mgus2"))

Read "mgus2.csv" with "," between the columns (guessed). To be sure, give separator=",".


## 1. Time and event from dates, with a competing event

The reference script makes dates from mgus2's months (follow-up from 2000-01-01; a month is 30.4375 days, as simplydata
counts); `add_time_and_event_columns(..., competing_with=["death_date"], unit="months")` must give back the vignette's
time and event for every row.

In [2]:
dates = sdata.table(cmp.data_file("mgus2_dates"))
for column in ("start", "end", "pcm_date", "death_date"):
    quiet(dates.turn_into_dates, column)
quiet(dates.add_time_and_event_columns, from_date="start", to_date="end", event_date="pcm_date", competing_with=["death_date"], unit="months")
made = Result(dates)
D = "mgus2 (survival), dates made from the times"
cmp.check_vector(D, 'add_time_and_event_columns (competing_with=["death_date"], unit="months")', "time (months)", made["time"])
cmp.check_vector(D, 'add_time_and_event_columns (competing_with=["death_date"], unit="months")', "event", made["event"])

Read "mgus2_dates.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  mgus2 (survival), dates made from the times | add_time_and_event_columns (competing_with=["death_date"], unit="months") | time (months) [1]: simplydata 29.9958932238, the vignette's etime, in whole days 29.9958932238; relative difference 0.00e+00 (allowed 1e-10)
      Source: the vignette's etime, in whole days (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/mgus2.html
agrees  mgus2 (survival), dates made from the times | add_time_and_event_columns (competing_with=["death_date"], unit="months") | time (months) [2]: simplydata 25.0020533881, the vignette's etime, in whole days 25.0020533881; relative difference 0.00e+00 (allowed 1e-10)
      Source: the vignette's etime, in whole days (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/mgus2.html
agrees  mgus2 (survival), dates made from the times | add_time_and_event_columns (competing_w

## 2. The chance of PCM when death can come first; Gray's test

In [3]:
risk = quiet(mgus.find_risk_over_time, time="etime", event="event", value="pcm", competing_with="death", at_time_points=[60, 120, 240])
check = 'find_risk_over_time (competing_with="death", at_time_points=[60, 120, 240])'
for event in ("pcm", "death"):
    r = Result(risk, key="time", where={"event": event})
    for p in (60, 120, 240):
        cmp.check("mgus2 (survival)", check, f"risk of {event} by {p} months", r[p, "risk"])
        if event == "pcm":
            cmp.check_all("mgus2 (survival)", check, {f"pcm by {p} months: 95 % low": r[p, "95 % CI low"], f"pcm by {p} months: 95 % high": r[p, "95 % CI high"]})
gray = Result(quiet(mgus.compare_survival, time="etime", event="event", value="pcm", competing_with="death", for_each_group_in="sex")).row(1)
cmp.check_all("mgus2 (survival)", 'compare_survival (competing_with="death", for_each_group_in="sex")', {"Gray's chi-square": gray["statistic"], "p-value": gray["p-value"]})

agrees  mgus2 (survival) | find_risk_over_time (competing_with="death", at_time_points=[60, 120, 240]) | risk of pcm by 60 months: simplydata 0.0341037129743, R survival survfit (Aalen-Johansen; vignette compete) (survival 3.8.12) 0.0341037129743; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survival survfit (Aalen-Johansen; vignette compete) (survival 3.8.12) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survfit.formula.html
agrees  mgus2 (survival) | find_risk_over_time (competing_with="death", at_time_points=[60, 120, 240]) | pcm by 60 months: 95 % low: simplydata 0.0257495519743, R survival survfit (Aalen-Johansen; vignette compete) (survival 3.8.12) 0.0257495519743; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survival survfit (Aalen-Johansen; vignette compete) (survival 3.8.12) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survfit.formula.html
agrees  mgus2 (survival) | find_risk_over_time (co

## 3. Cause-specific and Fine-Gray models; the Cox model on lung

In [4]:
for test_method in ("cox", "fine-gray"):
    f = Result(quiet(mgus.explain_survival, time="etime", event="event", value="pcm", competing_with="death", using=["age", "sex"], test_method=test_method),
               key="term")
    cmp.check_all("mgus2 (survival)", f'explain_survival (competing_with="death", test_method="{test_method}")',
                  {"age coefficient": f["age", "coefficient"], "sex coefficient": f["sex: M vs F", "coefficient"],
                   "age standard error": f["age", "standard error"], "sex standard error": f["sex: M vs F", "standard error"]})
lung = sdata.table(cmp.data_file("lung_complete"))
cox = Result(quiet(lung.explain_survival, time="time", event="status", value=2, using=["age", "sex", "ph.ecog"], test_method="cox"), key="term")
cmp.check_all("lung (survival)", 'explain_survival (test_method="cox")',
              {**{f"{t} coefficient": cox[t, "coefficient"] for t in ("age", "sex", "ph.ecog")},
               **{f"{t} standard error": cox[t, "standard error"] for t in ("age", "sex", "ph.ecog")}})

agrees  mgus2 (survival) | explain_survival (competing_with="death", test_method="cox") | age coefficient: simplydata 0.0130385698371, R survival coxph (death as censoring) (survival 3.8.12) 0.013038569825; relative difference 9.29e-10 (allowed 1e-06)
      Source: R survival coxph (death as censoring) (survival 3.8.12) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/coxph.html
agrees  mgus2 (survival) | explain_survival (competing_with="death", test_method="cox") | sex coefficient: simplydata -0.0251377892666, R survival coxph (death as censoring) (survival 3.8.12) -0.0251377892943; relative difference 1.10e-09 (allowed 1e-06)
      Source: R survival coxph (death as censoring) (survival 3.8.12) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/coxph.html
agrees  mgus2 (survival) | explain_survival (competing_with="death", test_method="cox") | age standard error: simplydata 0.00825868591211, R survival coxph (death as censoring) (survi

agrees  mgus2 (survival) | explain_survival (competing_with="death", test_method="fine-gray") | age coefficient: simplydata -0.0173381534947, R cmprsk crr (cmprsk 2.2.12) -0.0173381534947; relative difference 0.00e+00 (allowed 1e-06)
      Source: R cmprsk crr (cmprsk 2.2.12) (run reference) - https://search.r-project.org/CRAN/refmans/cmprsk/html/crr.html
agrees  mgus2 (survival) | explain_survival (competing_with="death", test_method="fine-gray") | sex coefficient: simplydata -0.260038237884, R cmprsk crr (cmprsk 2.2.12) -0.260038237884; relative difference 0.00e+00 (allowed 1e-06)
      Source: R cmprsk crr (cmprsk 2.2.12) (run reference) - https://search.r-project.org/CRAN/refmans/cmprsk/html/crr.html
agrees  mgus2 (survival) | explain_survival (competing_with="death", test_method="fine-gray") | age standard error: simplydata 0.00573710323024, R cmprsk crr (cmprsk 2.2.12) 0.00573710323024; relative difference 0.00e+00 (allowed 1e-06)
      Source: R cmprsk crr (cmprsk 2.2.12) (run r

## 4. Parametric models: Weibull and log-normal (ovarian, the help page of `survreg`)

In [5]:
ovarian = sdata.table(cmp.data_file("ovarian"))
for distribution in ("weibull", "log-normal"):
    f = Result(quiet(ovarian.explain_survival, time="futime", event="fustat", using=["ecog.ps", "rx"], distribution=distribution), key="term")
    cmp.check_all("ovarian (survival)", f'explain_survival (distribution="{distribution}")',
                  {"ecog.ps coefficient": f["ecog.ps", "coefficient"], "rx coefficient": f["rx", "coefficient"],
                   "ecog.ps standard error": f["ecog.ps", "standard error"], "rx standard error": f["rx", "standard error"]})

Read "ovarian.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  ovarian (survival) | explain_survival (distribution="weibull") | ecog.ps coefficient: simplydata -0.38504254935, R survival survreg (help page ovarian) (survival 3.8.12) -0.38504254935; relative difference 0.00e+00 (allowed 1e-06)
      Source: R survival survreg (help page ovarian) (survival 3.8.12) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survreg.html
agrees  ovarian (survival) | explain_survival (distribution="weibull") | rx coefficient: simplydata 0.528645496261, R survival survreg (help page ovarian) (survival 3.8.12) 0.528645496261; relative difference 0.00e+00 (allowed 1e-06)
      Source: R survival survreg (help page ovarian) (survival 3.8.12) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survreg.html
agrees  ovarian (survival) | explain_survival (distribution="weibull") | ecog.ps standard error: simplydata 0.527001781436, R survival survreg (help page ovarian) (survival 3.8.12) 0.527001781436; relative diffe

## 5. Restricted mean survival time; adjusted curves

In [6]:
full_lung = sdata.table(FETCHED + "lung.csv", separator=",")
rm = Result(quiet(full_lung.compare_survival, time="time", event="status", value=2, for_each_group_in="sex", average_time_up_to=365)).row(1)
cmp.check_all("lung (survival)", "compare_survival (average_time_up_to=365)",
              {"average time up to 365 days, sex 1": rm["average time in 1"], "average time up to 365 days, sex 2": rm["average time in 2"],
               "difference (2 minus 1)": rm["difference in average time"], "ratio (2 over 1)": rm["ratio of average times"]})
adjusted = quiet(sdata.table(cmp.data_file("lung_sex_group")).find_survival_over_time, time="time", event="status", value=2, for_each_group_in="sex_group",
                 allowing_for=["age"], at_time_points=[180, 365])
a = Result(adjusted)
survival_column = [c for c in a.columns if c.startswith("survival") and "CI" not in c][0]
for i in range(1, len(a) + 1):
    row = a.row(i)
    cmp.check("lung (survival)", 'find_survival_over_time (allowing_for=["age"], at_time_points=[180, 365])', f"{row[a.columns[0]]}, day {row['time']:g}",
              row[survival_column])

agrees  lung (survival) | compare_survival (average_time_up_to=365) | average time up to 365 days, sex 1: simplydata 241.495085192, R survRM2 rmst2 (survRM2 1.0.5) 241.495085192; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survRM2 rmst2 (survRM2 1.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/survRM2/html/rmst2.html
agrees  lung (survival) | compare_survival (average_time_up_to=365) | average time up to 365 days, sex 2: simplydata 297.465409532, R survRM2 rmst2 (survRM2 1.0.5) 297.465409532; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survRM2 rmst2 (survRM2 1.0.5) (run reference) - https://search.r-project.org/CRAN/refmans/survRM2/html/rmst2.html
agrees  lung (survival) | compare_survival (average_time_up_to=365) | difference (2 minus 1): simplydata 55.97032434, R survRM2 rmst2 (survRM2 1.0.5) 55.97032434; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survRM2 rmst2 (survRM2 1.0.5) (run reference) - https://search.r-pro

agrees  lung (survival) | find_survival_over_time (allowing_for=["age"], at_time_points=[180, 365]) | female, day 180: simplydata 0.790374898921, R adjustedCurves adjustedsurv (direct) (adjustedCurves 0.12.0) 0.790374898921; relative difference 0.00e+00 (allowed 1e-06)
      Source: R adjustedCurves adjustedsurv (direct) (adjustedCurves 0.12.0) (run reference) - https://search.r-project.org/CRAN/refmans/adjustedCurves/html/adjustedsurv.html
agrees  lung (survival) | find_survival_over_time (allowing_for=["age"], at_time_points=[180, 365]) | female, day 365: simplydata 0.522746050021, R adjustedCurves adjustedsurv (direct) (adjustedCurves 0.12.0) 0.522746050021; relative difference 0.00e+00 (allowed 1e-06)
      Source: R adjustedCurves adjustedsurv (direct) (adjustedCurves 0.12.0) (run reference) - https://search.r-project.org/CRAN/refmans/adjustedCurves/html/adjustedsurv.html
agrees  lung (survival) | find_survival_over_time (allowing_for=["age"], at_time_points=[180, 365]) | male, da

## 6. Cox model with multiple imputation (lung; wt.loss has 14 empty cells)

simplydata fills the empty cells 20 times (statsmodels MICEData) and pools 20 Cox fits; against R mice (m = 400) +
`pool(coxph)` within 4 Monte Carlo standard errors, and a rerun with the same seed.

In [7]:
mi = Result(quiet(full_lung.explain_survival, time="time", event="status", value=2, using=["age", "ph.ecog", "wt.loss"], empty_cells="multiple imputation"),
            key="term")
again = Result(quiet(full_lung.explain_survival, time="time", event="status", value=2, using=["age", "ph.ecog", "wt.loss"],
                     empty_cells="multiple imputation", with_seed=1), key="term")
for c in ("age", "ph.ecog", "wt.loss"):
    cmp.check("lung (survival)", 'explain_survival (empty_cells="multiple imputation", with_seed=)', f"{c} pooled coefficient", mi[c, "coefficient"])
    cmp.check_equal("lung (survival)", 'explain_survival (empty_cells="multiple imputation", with_seed=)', f"{c} coefficient, rerun with with_seed=1",
                    mi[c, "coefficient"], again[c, "coefficient"], "simplydata, same seed")

agrees  lung (survival) | explain_survival (empty_cells="multiple imputation", with_seed=) | age pooled coefficient: simplydata 0.0106939497721, R mice (m = 400) + pool(coxph), within 4 MCSE (mice 3.19.0) 0.0107638666014; difference 6.99e-05 (allowed 2e-04)
      Source: R mice (m = 400) + pool(coxph), within 4 MCSE (mice 3.19.0) (run reference) - https://search.r-project.org/CRAN/refmans/mice/html/pool.html
agrees  lung (survival) | explain_survival (empty_cells="multiple imputation", with_seed=) | age coefficient, rerun with with_seed=1: simplydata 0.0106939497721, simplydata, same seed 0.0106939497721; relative difference 0.00e+00 (allowed 1e-12)
      Source: simplydata, same seed
agrees  lung (survival) | explain_survival (empty_cells="multiple imputation", with_seed=) | ph.ecog pooled coefficient: simplydata 0.483941726386, R mice (m = 400) + pool(coxph), within 4 MCSE (mice 3.19.0) 0.480882439912; difference 3.06e-03 (allowed 1e-02)
      Source: R mice (m = 400) + pool(coxph), 

## 7. The numbers the survival charts draw: `chart.get_numbers_drawn()` (R `survfit`, adjustedCurves)

Each chart gives back what it drew as a table (layer, kind, series, x, y); the curves are its lines, one per colour
group (the censor marks are dots). A curve starts at time 0 (survival 1, risk 0), steps at every time with an event and
runs on to the last time followed. Compared, step by step (time and height), with R `survfit` (Kaplan-Meier by sex on
lung; late entry with `Surv(entry, time, event)` on the lung rows followed after day 30; the Aalen-Johansen risk of PCM
with death competing on mgus2) and with R adjustedCurves `adjustedsurv(method = "direct")` for the curves adjusted
for age.

In [8]:
def curves(chart):
    """The lines a chart drew (chart.get_numbers_drawn()), as {series: (x values, y values)}, read with simplydata words."""
    lines = Result(quiet(chart.get_numbers_drawn), where={"kind": "line"})
    found = {}
    for name, x, y in zip(lines["series"], lines["x"], lines["y"]):
        xs, ys = found.setdefault(str(name), ([], []))
        xs.append(x)
        ys.append(y)
    return found


full_lung = sdata.table(FETCHED + "lung.csv", separator=",")
chart = full_lung.start_chart()
quiet(chart.draw_survival_curves, time="time", event="status", value=2, colour="sex")
for sex, (times, heights) in curves(chart).items():
    cmp.check_vector("lung (survival)", 'chart draw_survival_curves (colour="sex")', f"curve of sex {sex}: time at each step", times)
    cmp.check_vector("lung (survival)", 'chart draw_survival_curves (colour="sex")', f"curve of sex {sex}: survival at each step", heights)

chart = sdata.table(cmp.data_file("lung_after_day_30")).start_chart()
quiet(chart.draw_survival_curves, time="time", event="status", value=2, followed_from="entry")
(times, heights), = curves(chart).values()
cmp.check_vector("lung after day 30 (survival)", 'chart draw_survival_curves (followed_from="entry")', "curve: time at each step", times)
cmp.check_vector("lung after day 30 (survival)", 'chart draw_survival_curves (followed_from="entry")', "curve: survival at each step", heights)

chart = sdata.table(cmp.data_file("lung_sex_group")).start_chart()
quiet(chart.draw_survival_curves, time="time", event="status", value=2, colour="sex_group", allowing_for=["age"])
for group, (times, heights) in curves(chart).items():
    cmp.check_vector("lung (survival)", 'chart draw_survival_curves (allowing_for=["age"])', f"adjusted curve of {group}: time at each step", times)
    cmp.check_vector("lung (survival)", 'chart draw_survival_curves (allowing_for=["age"])', f"adjusted curve of {group}: survival at each step", heights)

chart = mgus.start_chart()
quiet(chart.draw_risk_curves, time="etime", event="event", value="pcm", competing_with="death")
(times, heights), = curves(chart).values()
cmp.check_vector("mgus2 (survival)", 'chart draw_risk_curves (competing_with="death")', "curve: time at each step", times)
cmp.check_vector("mgus2 (survival)", 'chart draw_risk_curves (competing_with="death")', "curve: risk of pcm at each step", heights)

agrees  lung (survival) | chart draw_survival_curves (colour="sex") | curve of sex 1: time at each step [1]: simplydata 0, R survival survfit (Kaplan-Meier) (survival 3.8.12) 0; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survival survfit (Kaplan-Meier) (survival 3.8.12) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survfit.formula.html
agrees  lung (survival) | chart draw_survival_curves (colour="sex") | curve of sex 1: time at each step [2]: simplydata 11, R survival survfit (Kaplan-Meier) (survival 3.8.12) 11; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survival survfit (Kaplan-Meier) (survival 3.8.12) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survfit.formula.html
agrees  lung (survival) | chart draw_survival_curves (colour="sex") | curve of sex 1: time at each step [3]: simplydata 12, R survival survfit (Kaplan-Meier) (survival 3.8.12) 12; relative difference 0.00e+00 (allowed 1e-10)
     

agrees  lung (survival) | chart draw_survival_curves (allowing_for=["age"]) | adjusted curve of female: time at each step [1]: simplydata 0, R adjustedCurves adjustedsurv (direct, Cox model) (adjustedCurves 0.12.0) 0; relative difference 0.00e+00 (allowed 1e-06)
      Source: R adjustedCurves adjustedsurv (direct, Cox model) (adjustedCurves 0.12.0) (run reference) - https://search.r-project.org/CRAN/refmans/adjustedCurves/html/adjustedsurv.html
agrees  lung (survival) | chart draw_survival_curves (allowing_for=["age"]) | adjusted curve of female: time at each step [2]: simplydata 5, R adjustedCurves adjustedsurv (direct, Cox model) (adjustedCurves 0.12.0) 5; relative difference 0.00e+00 (allowed 1e-06)
      Source: R adjustedCurves adjustedsurv (direct, Cox model) (adjustedCurves 0.12.0) (run reference) - https://search.r-project.org/CRAN/refmans/adjustedCurves/html/adjustedsurv.html
agrees  lung (survival) | chart draw_survival_curves (allowing_for=["age"]) | adjusted curve of female

agrees  lung (survival) | chart draw_survival_curves (allowing_for=["age"]) | adjusted curve of female: survival at each step [100]: simplydata 0.505047130148, R adjustedCurves adjustedsurv (direct, Cox model) (adjustedCurves 0.12.0) 0.505047130148; relative difference 0.00e+00 (allowed 1e-06)
      Source: R adjustedCurves adjustedsurv (direct, Cox model) (adjustedCurves 0.12.0) (run reference) - https://search.r-project.org/CRAN/refmans/adjustedCurves/html/adjustedsurv.html
agrees  lung (survival) | chart draw_survival_curves (allowing_for=["age"]) | adjusted curve of female: survival at each step [101]: simplydata 0.498977599003, R adjustedCurves adjustedsurv (direct, Cox model) (adjustedCurves 0.12.0) 0.498977599003; relative difference 0.00e+00 (allowed 1e-06)
      Source: R adjustedCurves adjustedsurv (direct, Cox model) (adjustedCurves 0.12.0) (run reference) - https://search.r-project.org/CRAN/refmans/adjustedCurves/html/adjustedsurv.html
agrees  lung (survival) | chart draw_s

In [9]:
cmp.finish()

30_survival_more: 3003 comparisons - 3003 agrees
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,30_survival_more,"mgus2 (survival), dates made from the times","add_time_and_event_columns (competing_with=[""d...",time (months) [1],29.995893,29.995893,"the vignette's etime, in whole days",0.0,1.000000e-10,relative,0.0,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
1,30_survival_more,"mgus2 (survival), dates made from the times","add_time_and_event_columns (competing_with=[""d...",time (months) [2],25.002053,25.002053,"the vignette's etime, in whole days",0.0,1.000000e-10,relative,0.0,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
2,30_survival_more,"mgus2 (survival), dates made from the times","add_time_and_event_columns (competing_with=[""d...",time (months) [3],45.995893,45.995893,"the vignette's etime, in whole days",0.0,1.000000e-10,relative,0.0,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
3,30_survival_more,"mgus2 (survival), dates made from the times","add_time_and_event_columns (competing_with=[""d...",time (months) [4],91.991786,91.991786,"the vignette's etime, in whole days",0.0,1.000000e-10,relative,0.0,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
4,30_survival_more,"mgus2 (survival), dates made from the times","add_time_and_event_columns (competing_with=[""d...",time (months) [5],8.016427,8.016427,"the vignette's etime, in whole days",0.0,1.000000e-10,relative,0.0,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2998,30_survival_more,mgus2 (survival),"chart draw_risk_curves (competing_with=""death"")",curve: risk of pcm at each step [212],0.117692,0.117692,R survival survfit (Aalen-Johansen; vignette c...,0.0,1.000000e-10,relative,0.0,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
2999,30_survival_more,mgus2 (survival),"chart draw_risk_curves (competing_with=""death"")",curve: risk of pcm at each step [213],0.134042,0.134042,R survival survfit (Aalen-Johansen; vignette c...,0.0,1.000000e-10,relative,0.0,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
3000,30_survival_more,mgus2 (survival),"chart draw_risk_curves (competing_with=""death"")",curve: risk of pcm at each step [214],0.161292,0.161292,R survival survfit (Aalen-Johansen; vignette c...,0.0,1.000000e-10,relative,0.0,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
3001,30_survival_more,mgus2 (survival),"chart draw_risk_curves (competing_with=""death"")",curve: risk of pcm at each step [215],0.161292,0.161292,R survival survfit (Aalen-Johansen; vignette c...,0.0,1.000000e-10,relative,0.0,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/surv...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
